In [1]:
import re, os
import pdfplumber
import pandas as pd

In [2]:
proxy = "http://cache.ha.univ-nantes.fr:3128/"

os.environ['http_proxy'] = proxy
os.environ['https_proxy'] = proxy
os.environ['HTTP_PROXY'] = proxy
os.environ['HTTPS_PROXY'] = proxy

In [3]:
# -----------------------------
# 1. Extraction texte PDF
# -----------------------------
def extract_text_from_pdf(pdf_path):
    text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + "\n"
    return text


# -----------------------------
# 2. Nettoyage robuste du texte
# -----------------------------
def clean_text(text):
    # supprime retours ligne / espaces multiples
    text = re.sub(r"\s+", " ", text)

    # corrige les négatifs cassés: "- 0.25" → "-0.25"
    text = re.sub(r"-\s+(\d)", r"-\1", text)

    return text


# -----------------------------
# 3. Extraction des scores
# -----------------------------
def extract_scores_from_text(text):
    text = clean_text(text)

    pattern = r"Naive:\s*([-+]?\d*\.?\d+).*?Cosine:\s*([-+]?\d*\.?\d+).*?NLI:\s*([-+]?\d*\.?\d+)"

    matches = re.findall(pattern, text)

    results = []
    for naive, cosine, nli in matches:
        results.append({
            "Naive": float(naive),
            "Cosine": float(cosine),
            "NLI": float(nli)
        })

    return results


# -----------------------------
# 4. Pipeline complet
# -----------------------------
def extract_scores_from_pdf(pdf_path, return_df=True):
    text = extract_text_from_pdf(pdf_path)

    # debug utile
    print("Occurrences 'Naive' dans le texte :", text.count("Naive"))

    scores = extract_scores_from_text(text)

    print("Nombre de triplets extraits :", len(scores))

    if return_df:
        return pd.DataFrame(scores)

    return scores


# -----------------------------
# 5. UTILISATION
# -----------------------------
pdf_path = "../data/json/metrics_fin.pdf"

df_scores = extract_scores_from_pdf(pdf_path)

Occurrences 'Naive' dans le texte : 364
Nombre de triplets extraits : 363


In [4]:
df_scores.shape

(363, 3)

In [5]:
df_scores.head()

,Naive,Cosine,NLI
0,0.0,1.0,0.333333
1,0.8,1.0,1.000000
2,0.8,1.0,0.600000
3,0.0,0.0,-1.000000
4,0.6,0.2,-0.600000


In [6]:
def build_dataset(df_scores):
    rows = []

    # vérifier cohérence
    assert len(df_scores) % 3 == 0, "Le nombre de lignes n'est pas multiple de 3"

    for i in range(0, len(df_scores), 3):
        r1 = df_scores.iloc[i]
        r2 = df_scores.iloc[i+1]
        r3 = df_scores.iloc[i+2]

        row = {
            "Naive1": r1["Naive"],
            "Naive2": r2["Naive"],
            "Naive3": r3["Naive"],

            "Cosine1": r1["Cosine"],
            "Cosine2": r2["Cosine"],
            "Cosine3": r3["Cosine"],

            "NLI1": r1["NLI"],
            "NLI2": r2["NLI"],
            "NLI3": r3["NLI"],
        }

        # calcul FactScore
        for k in [1, 2, 3]:
            row[f"FactScore{k}"] = (
                0.2 * row[f"Naive{k}"] +
                0.2 * row[f"Cosine{k}"] +
                0.6 * row[f"NLI{k}"]
            )

        rows.append(row)

    return pd.DataFrame(rows)

In [7]:
df_final = build_dataset(df_scores)

In [8]:
df_final.shape

(121, 12)

In [9]:
df_final.head()

,Naive1,Naive2,Naive3,Cosine1,Cosine2,Cosine3,NLI1,NLI2,NLI3,FactScore1,FactScore2,FactScore3
0,0.000000,0.800000,0.800000,1.000000,1.000000,1.000000,0.333333,1.000000,0.600000,0.400000,0.960000,0.720000
1,0.000000,0.600000,0.000000,0.000000,0.200000,0.000000,-1.000000,-0.600000,-1.000000,-0.600000,-0.200000,-0.600000
2,0.357143,1.000000,0.642857,0.642857,1.000000,0.785714,0.357143,0.600000,0.428571,0.414286,0.760000,0.542857
3,0.500000,1.000000,1.000000,1.000000,1.000000,0.750000,1.000000,0.500000,0.500000,0.900000,0.700000,0.650000
4,0.888889,0.888889,0.888889,0.888889,0.888889,0.666667,0.777778,0.444444,0.444444,0.822222,0.622222,0.577778


In [11]:
df_final.to_csv("../data/json/factscores.csv", index=False)